In [ ]:
import numpy as np
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Normalize, Resize, ToTensor, CenterCrop
from train_cuda import train

In [ ]:
# temp dataset
temp_transform = Compose([Resize(64), CenterCrop(64), ToTensor()])
temp_dataset = ImageFolder(root="./Datasets/flower_photos", transform=temp_transform)

In [ ]:
# No need to bother with shuffling, as this is not the data loader we’ll use to train the
# model anyway. We’ll use it to compute statistics only. By the way, we need
# statistics for each channel, as required by the Normalize() transform.
bs = 64
temp_loader = DataLoader(temp_dataset, batch_size=bs)

In [ ]:
# statistics of each image per channel
results = []
for images, labels in temp_loader:
    # NCHW
    n_samples, n_channels, n_height, n_width = images.size()
    # Flatten HW into a single dimension
    flatten_per_channel = images.reshape(n_samples, n_channels, -1)
    # Computes statistics of each image per channel
    # Average pixel value per channel
    # (n_samples, n_channels)
    means = flatten_per_channel.mean(dim=2)
    # Standard deviation of pixel values per channel
    # (n_samples, n_channels)
    stds = flatten_per_channel.std(dim=2)
    n_samples = torch.tensor([n_samples] * n_channels).float()
    # Adds up statistics of all images in a mini-batch
    # (1, n_channels)
    sum_means = means.sum(dim=0)
    sum_stds = stds.sum(dim=0)
    result = torch.stack([n_samples, sum_means, sum_stds])
    # Append results
    results.append(result)
results = torch.stack(results).sum(dim=0)


In [ ]:
total_samples, total_means, total_stds = results
norm_mean = total_means / total_samples
norm_std = total_stds / total_samples

### 核心数学问题

标准差**不是可加量**。

> 
> 单张图片 std 的平均值 ≠ 整个数据集全部像素的 std
> 这个算出来的 std 不能给 `Normalize` 使用！
> 这个代码只能用来看：**平均每张图片内部的像素波动有多大**，不是数据集全局像素分布的标准差。

In [ ]:
c = 3
sum_x  = torch.zeros(c)
sum_x2 = torch.zeros(c)
total_pixels = 0

for images, labels in temp_loader:
    N, C, H, W = images.shape
    flat = images.reshape(N, C, -1) # [N, C, HW]

    # 每个通道：本batch全部像素之和
    sum_x += flat.sum(dim=[0, 2])
    # 每个通道：本batch全部像素平方之和
    sum_x2 += (flat ** 2).sum(dim=[0, 2])
    total_pixels += N * H * W

mean = sum_x / total_pixels
var  = (sum_x2 / total_pixels) - mean ** 2
std  = torch.sqrt(var)

print("数据集全局mean：", mean)
print("数据集全局std：", std)


In [ ]:
mean = torch.tensor([0.4882, 0.4374, 0.3103])
std = torch.tensor([0.2832, 0.2543, 0.2812])

In [ ]:
size = 64
composer = Compose([Resize(size), CenterCrop(size), ToTensor(), Normalize(mean, std)])


In [ ]:
train_data = ImageFolder(root="./Datasets/flower_photos", transform=composer)
# Builds a loader of each set
bs = 64
train_loader = DataLoader(train_data, batch_size=bs, shuffle=True)

In [ ]:
class Flowercnn(nn.Module):
    def __init__(self, n_filters, p=0.0):
        super().__init__()
        self.n_filters = n_filters
        self.p = p
        # Creates the convolution layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=n_filters, kernel_size=3)
        self.conv2 = nn.Conv2d(
            in_channels=n_filters, out_channels=n_filters * 2, kernel_size=3
        )
        self.conv3 = nn.Conv2d(
            in_channels=n_filters * 2, out_channels=n_filters * 4, kernel_size=3
        )
        # Creates the linear layers
        self.fc1 = nn.Linear(n_filters * 4 * 6 * 6, 120)
        self.fc2 = nn.Linear(120, 50)
        self.fc3 = nn.Linear(50, 5)
        # Creates dropout layers
        self.drop = nn.Dropout(self.p)

    def featurizer(self, x):
        # First convolutional block
        # 3@64x64 -> n_filters@62x62 -> n_filters@31x31
        x = self.conv1(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        # Second convolutional block
        # n_filters@31x31 -> n_filters@29x29 -> n_filters@14x14
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        # Third convolutional block
        # n_filters@14x14 -> n_filters@12x12 -> n_filters@6x6
        x = self.conv3(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        # Input dimension (n_filters*4@6x6)
        # Output dimension (n_filters * 4 * 6 * 6)
        x = nn.Flatten()(x)
        return x

    def classifier(self, x):
        # Classifier
        # Hidden Layer
        # Input dimension (n_feature * 4 * 6 * 6)
        # Output dimension (50)
        if self.p > 0:
            x = self.drop(x)
        x = self.fc1(x)
        x = F.relu(x)
        # Output Layer
        # Input dimension (50)
        # Output dimension (5)
        if self.p > 0:
            x = self.drop(x)
        x = self.fc2(x)
        x = F.relu(x)
        if self.p > 0:
            x = self.drop(x)
        x = self.fc3(x)
        return x

    def forward(self, x):
        x = self.featurizer(x)
        x = self.classifier(x)
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Flowercnn(n_filters=12, p=0.3).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer_rps = optim.Adam(model.parameters(), lr=3e-4)
epochs = 50
lr = 3e-4


In [ ]:
losses, val_losses = train(
    model, train_loader, None, loss_fn, optimizer_rps, epochs, device
)

In [ ]:
torch.save(model.state_dict(), "./Datasets/flower.pth")

In [ ]:
from PIL import Image

# 加载png图片
img1 = Image.open("./pictures/sun.png").convert("RGB")  # convert RGB，避免png透明通道4通道问题
img2 = Image.open("./pictures/xcj.png").convert("RGB")
img = torch.stack([composer(img1), composer(img2)], dim=0)  # Add batch dimension
print(img.shape)


In [ ]:
model.load_state_dict(torch.load("./Datasets/flower.pth", map_location=device))
y = model(img.to(device))
y

In [ ]:
predicted_indices = y.argmax(dim=1).cpu().numpy()

name_list = np.array(["daisy", "dandelion", "rose", "sunflower", "tulip"])
print(name_list[predicted_indices])